# Phase 01.05 — Zero-shot vs LoRA comparison

Compares full-run predictions on identical frozen sample IDs, reports aggregate metrics and question-type slices, and writes the Phase 01 status artifact.


In [1]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path("/workspace/RoadBuddy")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault("CC", "/usr/bin/gcc")
os.environ.setdefault("CXX", "/usr/bin/g++")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from roadbuddy_common import *

os.chdir(PROJECT_ROOT)
seed_everything(SEED)
ensure_dirs()
print("Project root:", PROJECT_ROOT)
print("Model revision:", MODEL_REVISION)


Project root: /workspace/RoadBuddy
Model revision: b98f263eab246eb5269ade64edbdca8a887dc44d


In [2]:
candidate_runs = {
    "full": (
        PATHS.zero_shot / "full" / "predictions.csv",
        PATHS.lora / "full" / "evaluation" / "validation_predictions.csv",
    ),
    "debug20": (
        PATHS.zero_shot / "debug20" / "predictions.csv",
        PATHS.lora / "debug20" / "evaluation" / "validation_predictions.csv",
    ),
}
RUN_NAME = next((name for name, paths in candidate_runs.items() if all(path.is_file() for path in paths)), None)
assert RUN_NAME is not None, f"No matching zero-shot/LoRA prediction pair found: {candidate_runs}"
ZERO_PATH, LORA_PATH = candidate_runs[RUN_NAME]

zero = pd.read_csv(ZERO_PATH)
lora = pd.read_csv(LORA_PATH)
assert zero.sample_id.is_unique and lora.sample_id.is_unique
assert set(zero.sample_id) == set(lora.sample_id), "Runs must contain identical validation sample IDs"
print("Comparison run:", RUN_NAME)
print("Rows:", len(zero))

Comparison run: debug20
Rows: 20


## Aggregate and paired results


In [3]:
def summarize(name, frame):
    predictions = frame.prediction.fillna("INVALID")
    return {
        "experiment": name,
        "rows": len(frame),
        "accuracy": accuracy_score(frame.answer, predictions),
        "macro_f1": f1_score(frame.answer, predictions, labels=list(CHOICES), average="macro", zero_division=0),
        "parse_rate": frame.prediction.notna().mean(),
    }

comparison = pd.DataFrame([summarize("zero_shot_f1", zero), summarize("lora_r16_f1", lora)])
paired = zero[["sample_id", "answer", "question_type", "prediction", "correct"]].merge(
    lora[["sample_id", "prediction", "correct"]], on="sample_id", suffixes=("_zero", "_lora"), validate="one_to_one"
)
paired["transition"] = paired.apply(lambda row: ("wrong→right" if not row.correct_zero and row.correct_lora else "right→wrong" if row.correct_zero and not row.correct_lora else "unchanged"), axis=1)
display(comparison)
display(paired.transition.value_counts())


,experiment,rows,accuracy,macro_f1,parse_rate
0,zero_shot_f1,20,0.65,0.598485,1.0
1,lora_r16_f1,20,0.60,0.569444,1.0


transition
unchanged      19
right→wrong     1
Name: count, dtype: int64

## Question-type slices and final artifacts


In [4]:
by_type = []
for name, frame in (("zero_shot_f1", zero), ("lora_r16_f1", lora)):
    for qtype, group in frame.groupby("question_type", dropna=False):
        by_type.append({"experiment": name, "question_type": qtype, "rows": len(group), "accuracy": group.correct.mean()})
by_type = pd.DataFrame(by_type)

out_dir = PATHS.phase1_output
comparison.to_csv(out_dir / "phase01_comparison.csv", index=False)
by_type.to_csv(out_dir / "phase01_by_question_type.csv", index=False)
paired.to_csv(out_dir / "phase01_paired_predictions.csv", index=False)

best = comparison.sort_values(["accuracy", "macro_f1"], ascending=False).iloc[0]
status = {
    "phase": "01", "status": "complete" if RUN_NAME == "full" else "debug_complete",
    "run_name": RUN_NAME, "validation_rows": len(paired),
    "winner": best.experiment, "winner_accuracy": float(best.accuracy),
    "model_revision": MODEL_REVISION, "seed": SEED,
    "frozen_validation_ids": str(PATHS.phase1_split / "validation_sample_ids.json"),
}
save_json(out_dir / "PHASE01_STATUS.json", status)
display(by_type)
display(status)
print("Saved comparison artifacts:", out_dir)

,experiment,question_type,rows,accuracy
0,zero_shot_f1,unknown,20,0.65
1,lora_r16_f1,unknown,20,0.60


{'phase': '01',
 'status': 'debug_complete',
 'run_name': 'debug20',
 'validation_rows': 20,
 'winner': 'zero_shot_f1',
 'winner_accuracy': 0.65,
 'model_revision': 'b98f263eab246eb5269ade64edbdca8a887dc44d',
 'seed': 42,
 'frozen_validation_ids': '/workspace/RoadBuddy/data/splits/phase01/validation_sample_ids.json'}

Saved comparison artifacts: /workspace/RoadBuddy/outputs/phase01


## Nhận xét tổng hợp sau lần chạy Phase 01.05

**Trạng thái:** `debug_complete`, run `debug20`. Toàn bộ pipeline Phase01 hoạt động end-to-end, nhưng kết quả chưa phải full experiment.

### So sánh cuối

| Mô hình | Rows | Accuracy | Macro-F1 | Parse rate |
|---|---:|---:|---:|---:|
| Zero-shot F1 | 20 | 0.6500 | 0.5985 | 1.0000 |
| LoRA r16 F1 | 20 | 0.6000 | 0.5694 | 1.0000 |

Paired transitions: 12 mẫu cả hai cùng đúng, 7 mẫu cả hai cùng sai, 1 mẫu zero-shot đúng nhưng LoRA sai, 0 mẫu LoRA sửa được lỗi zero-shot. Chỉ 1/20 prediction thay đổi sau fine-tune debug.

### Kết luận có thể khẳng định

- Data contract, split chống video leakage, model loading, zero-shot inference, LoRA training, adapter loading và evaluation đều chạy được.
- Zero-shot là winner của debug run với lợi thế 0,05 accuracy và khoảng 0,029 macro-F1.
- LoRA debug chưa mang lại lợi ích; không nên dùng adapter này cho public-test submission.
- Parse rate 1.0 ở cả hai run cho thấy output-format contract ổn định.

### Điều chưa thể kết luận

- Không thể kết luận hiệu năng full-validation hoặc khả năng tổng quát từ 20 mẫu.
- Không thể phân tích theo loại câu hỏi vì tất cả rows đang là `unknown`.
- Không thể đánh giá public test vì không có nhãn.
- Không thể kết luận rank 16 hay LoRA nói chung không hiệu quả từ 2 optimizer steps.

### Ưu tiên tiếp theo

1. Bổ sung hoặc suy luận `question_type` có kiểm định.
2. Hoàn thiện training loop cho full run, đặc biệt partial gradient accumulation và provenance logging.
3. Chạy zero-shot và LoRA trên đủ 298 frozen validation rows.
4. Báo cáo confusion matrix, per-class F1, confidence interval hoặc bootstrap paired delta.
5. Chỉ tạo public-test submission sau khi chọn checkpoint bằng frozen validation, tuyệt đối không dựa vào public-test labels.